# 08 - Debugging the Axis Swap: A Guided Investigation

This notebook contains a deliberately buggy IQ-processing pipeline. Your job is to find the bug.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## The Setup

We have a two-channel IQ system (e.g., a dual-antenna receiver). The data is stored as `[samples, channels]` where channel 0 = I, channel 1 = Q.

In [ ]:
# Generate two-channel IQ data
np.random.seed(42)
n_samples = 1024

# Create [samples, 2] array: column 0 = I, column 1 = Q
t = np.arange(n_samples)
i_channel = 1.0 * np.cos(2 * np.pi * 0.05 * t) + 0.1 * np.random.randn(n_samples)
q_channel = 1.0 * np.sin(2 * np.pi * 0.05 * t) + 0.1 * np.random.randn(n_samples)

iq_data = np.column_stack([i_channel, q_channel])
print(f"iq_data shape: {iq_data.shape}")
print(f"iq_data dtype: {iq_data.dtype}")
print(f"First sample: I={iq_data[0, 0]:.4f}, Q={iq_data[0, 1]:.4f}")

## The Buggy Pipeline

The following code processes the data but contains an axis-related bug. Run it and see what happens.

In [ ]:
# BUGGY CODE - DO NOT TRUST THE RESULTS
def buggy_process(data):
    # "Optimize" by transposing first
    processed = data.T  # <-- This swaps axes!
    
    # Compute power for each channel
    power_ch0 = np.mean(processed[0] ** 2)
    power_ch1 = np.mean(processed[1] ** 2)
    
    return processed, power_ch0, power_ch1

processed, p0, p1 = buggy_process(iq_data)
print(f"Processed shape: {processed.shape}")
print(f"Power channel 0: {p0:.6f}")
print(f"Power channel 1: {p1:.6f}")

In [ ]:
# Let's also try plotting - something looks wrong
plt.figure(figsize=(10, 4))
plt.plot(processed[0], label='Channel 0')
plt.plot(processed[1], label='Channel 1')
plt.legend()
plt.title('Processed Channels')
plt.xlabel('Index')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.show()
print("Does this look like I and Q components of a clean signal?")

## Diagnostic Questions

1. What is the shape of the original `iq_data`? What should `processed` be?
2. What are the first few values of `iq_data[:, 0]` (I channel) vs `processed[0]`?
3. Compare the power values. Are they what you expected?

In [ ]:
# Step 1: Check shapes
print(f"Original shape: {iq_data.shape}")
print(f"After .T shape: {iq_data.T.shape}")
print(f"After .T: rows are now columns!")

In [ ]:
# Step 2: Compare values
print("Original I channel (first 5):", iq_data[:5, 0])
print("Processed[0] (first 5):      ", processed[0, :5])
print(f"\nAre they the same? {np.allclose(iq_data[:5, 0], processed[0, :5])}")

In [ ]:
# Step 3: Direct comparison
direct_power_i = np.mean(iq_data[:, 0] ** 2)
direct_power_q = np.mean(iq_data[:, 1] ** 2)
print(f"Direct I power: {direct_power_i:.6f}")
print(f"Direct Q power: {direct_power_q:.6f}")
print(f"Buggy Ch0 power: {p0:.6f}")
print(f"Buggy Ch1 power: {p1:.6f}")
print(f"\nDo they match? I: {np.isclose(direct_power_i, p0)}, Q: {np.isclose(direct_power_q, p1)}")

## Finding the Bug

**What went wrong?**

The `.T` transpose operation swaps axes:
- Original: shape `(1024, 2)` — rows=samples, cols=channels
- Transposed: shape `(2, 1024)` — rows=channels, cols=samples

After transposing, `processed[0]` is the entire first row of the transposed matrix (1024 values), which is actually the first *column* of the original data — but the indexing semantics have changed. The code treats `processed[0]` as "channel 0" when it's actually a different slice of data.

## The Fix

Don't transpose the data unnecessarily. Work with the original `[samples, channels]` layout.

In [ ]:
def correct_process(data):
    """Process IQ data correctly without transposing."""
    # Keep data in [samples, channels] layout
    i_component = data[:, 0]
    q_component = data[:, 1]
    
    power_i = np.mean(i_component ** 2)
    power_q = np.mean(q_component ** 2)
    
    return i_component, q_component, power_i, power_q

i_correct, q_correct, p_i, p_q = correct_process(iq_data)
print(f"Correct I power: {p_i:.6f}")
print(f"Correct Q power: {p_q:.6f}")
print(f"Matches direct:  {np.isclose(p_i, direct_power_i) and np.isclose(p_q, direct_power_q)}")

In [ ]:
# Corrected plot
plt.figure(figsize=(10, 4))
plt.plot(i_correct, label='I (correct)')
plt.plot(q_correct, label='Q (correct)')
plt.legend()
plt.title('Correctly Extracted I/Q Components')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.show()

## Reflection

In your own words, explain:
1. What was the bug?
2. Why did the bug cause incorrect results?
3. How could you prevent this type of bug in the future?

Write your answers below.

**Your answers:**

1. 

2. 

3. 

## Summary

Key lessons:
- Always check `.shape` when debugging array operations
- Compare a few sample values between original and processed data
- Use independent calculations (direct vs. processed) to verify results
- Transposing changes the semantic meaning of axes — be explicit about axis conventions
- Document your array layout: `[samples, channels]` vs `[channels, samples]`